In [ ]:
%reload_ext autoreload
%autoreload 2

from tqdm import trange
from flygym.compose import ActuatorType

import importlib
import miniproject.simulation
importlib.reload(miniproject.simulation)

from miniproject.simulation import MiniprojectSimulation
from submission.controller import Controller
from submission.controller import Controller, add_state_overlay

import mediapy
#import cv2
import matplotlib.pyplot as plt

sim = MiniprojectSimulation(level=4, seed=1)
print(sim.enable_wind)  # should print True
controller = Controller(sim)

actual_wind_angles = []

from flygym.compose import ActuatorType
import numpy as np


for _ in trange(60000): # 50 000 steps pour atteindre la cible sur flat
    joint_angles, adhesion = controller.step(sim)
    sim.set_actuator_inputs(sim.fly.name, ActuatorType.POSITION, joint_angles)
    sim.set_actuator_inputs(sim.fly.name, ActuatorType.ADHESION, adhesion)
    sim.step()
    sim.render_as_needed()
    actual_wind_angles.append(getattr(sim, 'current_wind_angle', None))


n_frames = len(sim.renderer.frames["birdeyecam"])
n_steps = len(controller.trajectory_states)
step_ratio = n_steps // n_frames

annotated = add_state_overlay(
    sim.renderer.frames["birdeyecam"],
    controller.trajectory_states,
    step_ratio,
    actual_wind_angles=actual_wind_angles,
    perceived_wind_angles=controller.trajectory_wind_perceived,
    headings=controller.trajectory_heading
)


mediapy.show_video(annotated, fps=sim.renderer.output_fps, title="top-down view with state")
mediapy.show_video(controller.frames, fps=sim.renderer.output_fps, title="ommatidia vision")
mediapy.show_video(controller.frames_drag_new, fps=sim.renderer.output_fps, title="ommatidia vision")

sim.renderer.show_in_notebook()

controller.plot_trajectory("ma_trajectoire.png")
controller.plot_trajectory_with_states("trajectory_states.png")


Failed to read module file 'C:\Users\adche\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\functools.py' for module 'functools': UnicodeDecodeError
Traceback (most recent call last):
  File "c:\Users\adche\OneDrive\Bureau\COURS\MASTER\CONTROLLING BEHAVIOUR IN ANIMALS AND ROBOTS\Controlling_BAR_Project\.venv\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\adche\OneDrive\Bureau\COURS\MASTER\CONTROLLING BEHAVIOUR IN ANIMALS AND ROBOTS\Controlling_BAR_Project\.venv\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\adche\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\importlib\__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

False


100%|██████████| 30000/30000 [01:09<00:00, 433.83it/s]


top-down view with state This browser does not support the video tag.


ommatidia vision This browser does not support the video tag.


ommatidia vision This browser does not support the video tag.


nmf/backcam This browser does not support the video tag.


birdeyecam This browser does not support the video tag.


📊 Trajectoire sauvegardée : ma_trajectoire.png
   - Points de trajectoire : 30000


In [ ]:
plt.imshow(controller.frames_drag_new[-1])